# Магия Бустингов (XGBoost & CatBoost)

**Цель семинара:**
- Понять физический смысл гиперпараметров градиентного бустинга.
- Увидеть на практике различия XGBoost и CatBoost.
- Научиться автоматическому подбору параметров (Tuning).


**Важное замечание:** Классификация vs Регрессия

Сегодня мы решаем задачу бинарной классификации (0 или 1), поэтому используем `XGBClassifier` и `CatBoostClassifier`. Если у вас задача регрессии (предсказание числа), логика остается той же самой, но:
- Используйте классы `XGBRegressor` / `CatBoostRegressor`.
- В качестве метрики используйте MSE/MAE/RMSE вместо LogLoss/AUC.
- Зависимости (графики), которые мы построим ниже, будут выглядеть аналогично.

In [ ]:
# !pip install xgboost catboost scikit-learn pandas matplotlib seaborn

## 1. Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

# Импорт моделей
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')
sns.set(style="whitegrid")


ДЕЛАЕМ ТРАГЕТ TODO

In [ ]:
print("Загрузка данных...")
data = fetch_openml(name='adult', version=2, as_frame=True)
X = data.data
y = data.target

# Таргет в 0 и 1
y = y.apply(lambda x: 1 if '>50K' in str(x) else 0)

# Заполняем пропуски
X = X.fillna(X.mode().iloc[0])

**ВАЖНЫЙ МОМЕНТ**

Нам нужны ДВЕ версии данных:
1. X_enc (Encoded): Все категории превращены в числа (0, 1, 2...). Нужно для XGBoost.
2. X_raw (Raw): Категории оставлены строками/category. Нужно для CatBoost (через cat_features).

In [ ]:

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Для CatBoost нам нужны имена или индексы категориальных колонок
cat_features_names = cat_cols

# Версия для XGBoost (Label Encoding)
X_enc = X.copy()
for col in cat_cols:
    le = LabelEncoder()
    X_enc[col] = le.fit_transform(X[col].astype(str))

# Версия для CatBoost (оставляем как есть, но приводим к string для стабильности)
X_raw = X.copy()
for col in cat_cols:
    X_raw[col] = X_raw[col].astype(str)

print("Данные готовы!")
print(f"Категориальные признаки: {cat_cols}")


In [ ]:
# ЗАДАНИЕ: Разбейте данные на Train и Validation (25% на валидацию).
# Важно: используйте random_state=42, чтобы индексы в обоих сплитах совпали!

# 1. Разбиваем X_enc (для XGBoost)
X_train_enc, X_val_enc, y_train, y_val = train_test_split(X_enc, y, test_size=0.25, random_state=42)

# 2. Разбиваем X_raw (для CatBoost) - y_train и y_val те же самые
X_train_raw, X_val_raw, _, _ = ... # ВАШ КОД ЗДЕСЬ

Для исследования гипперпараметров вам потребуется следующая функция. Внимательно ее изучите, так как в ней находится основная логика работы с любыми моделями классического машинного обучения.

In [ ]:
def plot_validation_curve(model_class, X_train, y_train, X_val, y_val, param_name, param_range, model_name="Model", fit_params=None, **model_params):
    """
    fit_params: словарь параметров, которые передаются в метод .fit() (например, cat_features)
    model_params: параметры, которые передаются при инициализации модели (например, depth, learning_rate)
    """
    if fit_params is None:
        fit_params = {}
        
    train_scores = []
    val_scores = []
    
    print(f"[{model_name}] Исследуем: {param_name}...")
    
    for value in param_range:
        current_params = {param_name: value}
        current_params.update(model_params)
        
        # Инициализация
        model = model_class(**current_params)
        
        # Обучение с передачей специфичных параметров (типа cat_features)
        try:
            # на случай если модель не поддерживает verbose (Выводить ли логи во время обучения)
            model.fit(X_train, y_train, verbose=False, **fit_params)
        except TypeError:
            # Если вдруг verbose не поддерживается или модель sklearn
            model.fit(X_train, y_train, **fit_params)
            
        # Предсказание
        preds_train = model.predict_proba(X_train)[:, 1]
        preds_val = model.predict_proba(X_val)[:, 1]
        
        train_scores.append(roc_auc_score(y_train, preds_train))
        val_scores.append(roc_auc_score(y_val, preds_val))
    
    # Визуализация
    plt.figure(figsize=(10, 5))
    plt.plot(param_range, train_scores, marker='o', label='Train ROC-AUC')
    plt.plot(param_range, val_scores, marker='o', label='Validation ROC-AUC')
    plt.title(f'{model_name}: Dependecy on {param_name}')
    plt.xlabel(param_name)
    plt.ylabel('ROC-AUC')
    plt.legend()
    plt.grid(True)
    plt.show()


## 2. XGBoost: Классика жанра

## 2.1 Количество деревьев (n_estimators)

Как долго нужно учить модель?

In [ ]:
# ЗАДАНИЕ: Выберите диапазон деревьев. Например, от 10 до 500. Возьмите небольшое колличество (<7)

n_estimators_range = ... # ВАШ КОД ЗДЕСЬ

base_xgb_params = {'learning_rate': 0.1, 'random_state': 42, 'n_jobs': -1, 'use_label_encoder': False, 'eval_metric': 'logloss'}
# Используйте данные _enc (числовые).
plot_validation_curve(
    model_class=XGBClassifier, 
    X_train=X_train_enc, y_train=y_train, 
    X_val=X_val_enc, y_val=y_val,
    param_name='n_estimators', 
    param_range=n_estimators_range, 
    model_name='XGBoost',
    **base_xgb_params
)



**Вопрос:** Почему синяя линия (Train) стремится к 1.0, а оранжевая (Validation) останавливается?

### 2.2 Глубина деревьев (max_depth)

Насколько сложные зависимости мы хотим искать?

In [ ]:
depth_range = [2, 4, 6, 10, 15]

# ЗАДАНИЕ: Заполните пропуски
# plot_validation_curve(...) # ВАШ КОД ЗДЕСЬ

**Вопрос:** Обычно в Random Forest деревья очень глубокие. Почему в бустинге на графике мы видим ухудшение результата при глубине > 8-10?

### 2.3 Исследуйте и другие гипперпараметры
Более подробно можно изучить в официальной [документации](https://xgboost.readthedocs.io/en/stable/)

## 3. CatBoost: Глубокое погружение

У [CatBoost](https://catboost.ai/docs/en/) есть свои уникальные параметры. Давайте их исследуем.
### 3.1 Количество итераций (iterations)

В CatBoost параметр называется iterations, а не n_estimators.

In [ ]:
iters_range = [50, 200, 500, 1000]

# Словарь параметров для .fit()
fit_params_cat = {'cat_features': cat_features_names}

# plot_validation_curve(
#    CatBoostClassifier,
#    X_train_raw, y_train, X_val_raw, y_val,  # ! ИСПОЛЬЗУЕМ RAW ДАННЫЕ
#    ...,                                     # ! ЗАПОЛНИТЕ ОСТАЛЬНЫЕ АРГУМЕНТЫ
#    fit_params=fit_params_cat,               # ! ПЕРЕДАЕМ CAT_FEATURES
# )

Сравните график переобучения с XGBoost. CatBoost обычно "держит" переобучение дольше благодаря специальным методам работы с данными.

In [ ]:
depth_range = [4, 6, 8, 10]
times = []
scores = []

print("CatBoost Depth Analysis...")
for d in depth_range:
    start = time.time()
    
    # Модель
    model = CatBoostClassifier(iterations=200, depth=d, learning_rate=0.1, 
                               verbose=0, random_seed=42)
    
    # ЗАДАНИЕ: Обучите модель НА X_train_raw, ОБЯЗАТЕЛЬНО передав cat_features
    # model.fit(...) # ВАШ КОД ЗДЕСЬ
    
    elapsed = time.time() - start
    # Предсказываем тоже на Raw данных
    score = roc_auc_score(y_val, model.predict_proba(X_val_raw)[:, 1])
    
    times.append(elapsed)
    scores.append(score)
    print(f"Depth={d}: Time={elapsed:.2f}s, AUC={score:.4f}")

# Рисуем два графика: Качество и Время
fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'tab:orange'
ax1.set_xlabel('Depth')
ax1.set_ylabel('ROC-AUC', color=color)
ax1.plot(depth_range, scores, marker='o', color=color, label='AUC')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # вторая ось Y
color = 'tab:blue'
ax2.set_ylabel('Time (sec)', color=color)
ax2.plot(depth_range, times, marker='x', linestyle='--', color=color, label='Time')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('CatBoost: Trade-off между качеством (глубиной) и временем')
plt.show()

**Вопрос:** На глубине 10 время обучения резко подскочило. Оправдал ли себя прирост качества?

### 3.3 Регуляризация (l2_leaf_reg)

Исследуем, как L2-регуляризация влияет на CatBoost именно при работе с категориальными признаками (где риск переобучения высок).

In [ ]:
reg_range = [1e-1, 5e-1, 1, 5, 10,]

# ЗАДАНИЕ: Заполните вызов функции
# fit_params=fit_params_cat обязателен!
# Данные должны быть X_train_raw!

plot_validation_curve(...) # ВАШ КОД ЗДЕСЬ


## 4. Автоматический подбор гиперпараметров (Tuning)

Ручной перебор параметров (как мы делали выше, меняя их по одному) помогает понять физику процесса, но для получения лучшего результата нужно искать комбинации параметров.

Полный перебор (GridSearch) для бустингов занимает слишком много времени. Поэтому стандартом индустрии является случайный поиск — Random Search.
### 4.1. Подбор для XGBoost

Важно: Для XGBoost мы используем данные X_train_enc, где категории уже закодированы числами.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

# 1. Задаем сетку параметров (распределения)
xgb_params = {
    'learning_rate': uniform(0.01, 0.2),      # От 0.01 до 0.21
    'n_estimators': randint(50, 300),         # От 50 до 300 деревьев
    'max_depth': randint(3, 10),              # Глубина от 3 до 10
    'subsample': uniform(0.6, 0.4)            # Доля данных от 0.6 до 1.0
}

# 2. Создаем базовую модель
xgb_model = XGBClassifier(
    n_jobs=-1, 
    random_state=42, 
    use_label_encoder=False, 
    eval_metric='logloss'
)

# 3. Настраиваем поиск
rs_xgb = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_params,
    n_iter=15,               # Количество случайных комбинаций
    scoring='roc_auc',       # Целевая метрика
    cv=3,                    # Кросс-валидация на 3 фолда
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# 4. Запускаем на ENCODED данных (числовых)
print("Подбор параметров для XGBoost...")
# rs_xgb.fit(...) # ВАШ КОД ЗДЕСЬ: передайте X_train_enc и y_train

print(f"Лучшие параметры XGBoost: {rs_xgb.best_params_}")
print(f"Лучший ROC-AUC: {rs_xgb.best_score_:.4f}")


### 4.2. Подбор для CatBoost
Важно:

Мы используем X_train_raw (данные с исходными категориями).

Мы обязательно передаем параметр cat_features при создании модели, чтобы она знала, как их обрабатывать.

In [ ]:
# 1. Задаем сетку параметров
# Обратите внимание: у CatBoost параметры называются иначе (depth, iterations)
cat_params = {
    'learning_rate': uniform(0.01, 0.2),
    'iterations': randint(100, 400),
    'depth': randint(4, 10),
    'l2_leaf_reg': randint(1, 10)
}

# 2. Создаем базовую модель CatBoost
# ВАШ КОД ЗДЕСЬ: Инициализируйте CatBoostClassifier.
# Обязательно передайте аргумент cat_features=cat_cols (список имен или индексов категорий)
# Также установите verbose=0, чтобы не засорять вывод.
cat_model = ...



# 3. Настраиваем поиск
rs_cat = RandomizedSearchCV(
    ... # Ваш код здесь
)


# 4. Запускаем обучение
print("Start Tuning CatBoost (Raw Data)...")

# ВАЖНО: Передаем X_train_raw (сырые данные)
# ВАШ КОД ЗДЕСЬ: Запустите метод fit
rs_cat.fit(...)


print(f"Лучшие параметры CatBoost: {rs_cat.best_params_}")
print(f"Лучший ROC-AUC (CV): {rs_cat.best_score_:.4f}")


### 4.3. Финальное сравнение на валидации

Теперь возьмем лучшие найденные модели (best_estimator_) и проверим их на отложенной выборке, которую модели еще не видели.

In [ ]:
print("-" * 40)
print("Финальная проверка на отложенной выборке (X_val)")
print("-" * 40)

# 1. Проверяем XGBoost (на закодированных данных X_val_enc)
best_xgb = rs_xgb.best_estimator_
xgb_pred = best_xgb.predict_proba(X_val_enc)[:, 1]
xgb_score = roc_auc_score(y_val, xgb_pred)
print(f"XGBoost Tuned AUC:  {xgb_score:.4f}")

# 2. Проверяем CatBoost (на сырых данных X_val_raw)
best_cat = rs_cat.best_estimator_
cat_pred = best_cat.predict_proba(X_val_raw)[:, 1]
cat_score = roc_auc_score(y_val, cat_pred)
print(f"CatBoost Tuned AUC: {cat_score:.4f}")

print("-" * 40)
if cat_score > xgb_score:
    print("🏆 Победитель: CatBoost")
else:
    print("🏆 Победитель: XGBoost")


**Вопрос на подумать:** Сравните результаты автоматического подбора с результатами, которые мы получали в начале семинара при дефолтных настройках. Сильно ли удалось поднять качество? Стоила ли игра свеч?